In [1]:
#RAG FROM SCRATCH

# Importing the libraries

import bs4
from langchain import hub
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [8]:
import os
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = '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'

In [16]:
import getpass
import os

if not os.environ.get("GROQ_API_KEY"):
  os.environ["GROQ_API_KEY"] = "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"

from langchain_groq import ChatGroq

llm = ChatGroq(model="llama3-8b-8192")

In [17]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

c:\Users\deshp\anaconda3\envs\raglangchain\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
#Indexing and loading the documents

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content","post-title","post-header")
        )
    )
)

In [11]:
docs = loader.load()

In [12]:
#Splitting

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splits = text_splitter.split_documents(docs)


In [18]:
vectorstore = Chroma.from_documents(documents=splits, 
                                    embedding=HuggingFaceEmbeddings())

In [19]:
retriever = vectorstore.as_retriever()

In [20]:
#RETRIEVAL AND GENERATION

prompt = hub.pull("rlm/rag-prompt")

In [21]:
llm = ChatGroq(model="llama3-8b-8192")

In [36]:
def format_docs(docs):
    """Format docs into a string."""
    return "\n\n".join(doc.page_content for doc in docs)

In [37]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [38]:
response = rag_chain.invoke("What is Task Decomposition?")
print(response)

Task Decomposition is the process of breaking down a complex task into smaller, manageable subtasks or steps. This is typically done using simple prompting, task-specific instructions, or human inputs. The goal is to create a clear outline or structure for the task, making it easier to plan and execute.
